# The Depth Gap in Initial Sharpness

Under the $C_i \approx 0$ approximation the initial sharpness is
$$\lambda_\parallel^{\text{init}} = (1+n)\,(\|p\|^2)^{\alpha}, \qquad \alpha = \tfrac{n}{n+1},$$
and the final sharpness is $(1+n)$. The normalized gap is therefore
$$\text{gap} = (1+n)\big(1 - \mathbb{E}[(\|p\|^2)^{\alpha}]\big).$$

By independence the expectation factorizes exactly:
$$\mathbb{E}[(\|p\|^2)^{\alpha}] = \underbrace{\mathbb{E}[(\|w_1\|^2)^{\alpha}]}_{\text{first layer}}\cdot\big(\underbrace{\mathbb{E}[b^{2\alpha}]}_{\text{per scalar layer}}\big)^{n}.$$

This notebook evaluates the two factors directly for the **same initializations** used everywhere else on the site (Gaussian: $a_k\sim\mathcal N(0,1/d)$, $b_i\sim\mathcal N(0,1)$; uniform: $a_k\sim U[-\sqrt{3/d},\sqrt{3/d}]$, $b_i\sim U[-\sqrt3,\sqrt3]$), prints the tables, and writes `sharpness_gap.json` for the website. The scalar factor and the Gaussian first-layer factor have closed forms; the uniform first-layer factor uses high-sample Monte Carlo. Everything is checked against direct MC of the full product before writing.

## Configuration

In [7]:
WIDTHS = [2, 3, 4, 5, 6]      # input width d
DEPTHS = [1, 2, 3, 4, 5]      # number of scalar (hidden) layers n
KINDS  = ["gaussian", "uniform"]

MC_SAMPLES   = 4_000_000      # samples for the uniform first-layer factor
CHECK_SAMPLES = 2_000_000     # samples for the full-product cross-check
SEED = 20240601

import json, math
from math import lgamma, sqrt, pi
import numpy as np

## The two factors

**Scalar layer** $\mathbb{E}[b^{2\alpha}]$ for unit-variance $b$:
- Gaussian $b\sim\mathcal N(0,1)$: $\mathbb{E}[|b|^{2\alpha}] = 2^{\alpha}\,\Gamma(\alpha+\tfrac12)/\sqrt\pi$.
- Uniform $b\sim U[-\sqrt3,\sqrt3]$: $\mathbb{E}[|b|^{2\alpha}] = 3^{\alpha}/(2\alpha+1)$.

**First layer** $\mathbb{E}[(\|w_1\|^2)^{\alpha}]$, $\|w_1\|^2=\sum_{k=1}^d a_k^2$, $a_k$ variance $1/d$:
- Gaussian: $\|w_1\|^2 = \tfrac1d\chi^2_d$, so $\mathbb{E}[(\|w_1\|^2)^{\alpha}] = d^{-\alpha}\,2^{\alpha}\,\Gamma(\tfrac d2+\alpha)/\Gamma(\tfrac d2)$.
- Uniform: no clean closed form — high-sample Monte Carlo.

In [8]:
def Eb2a(kind, a):
    """E[b^{2a}] for a unit-variance scalar layer."""
    if kind == "gaussian":
        return (2.0**a) * math.exp(lgamma(a + 0.5) - 0.5*math.log(pi))
    return (3.0**a) / (2*a + 1)        # uniform on [-sqrt3, sqrt3]

def EWa(kind, d, a, rng=None):
    """E[(||w1||^2)^a], ||w1||^2 = sum of d variance-1/d terms."""
    if kind == "gaussian":
        return d**(-a) * 2.0**a * math.exp(lgamma(d/2 + a) - lgamma(d/2))
    # uniform: Monte Carlo
    rng = rng or np.random.default_rng(SEED)
    aw = sqrt(3.0/d)
    A = rng.uniform(-aw, aw, size=(MC_SAMPLES, d))
    return float(np.mean(np.sum(A*A, axis=1)**a))

def product_factor(kind, d, n, rng=None):
    """E[(||p||^2)^alpha] = E[W^alpha] * (E[b^{2alpha}])^n,  alpha = n/(n+1)."""
    a = n/(n+1)
    return EWa(kind, d, a, rng) * Eb2a(kind, a)**n

## Cross-check against direct Monte Carlo of the full product

Sample whole networks, form $\|p\|^2 = \|w_1\|^2\prod_i b_i^2$, and compare $\mathbb{E}[(\|p\|^2)^\alpha]$ to the factorized value. They should agree to MC precision (~$10^{-3}$).

In [9]:
def mc_product(kind, d, n, M=CHECK_SAMPLES, seed=1):
    a = n/(n+1)
    rng = np.random.default_rng(seed)
    if kind == "gaussian":
        w1 = rng.normal(0, sqrt(1.0/d), size=(M, d))
        b  = rng.normal(0, 1, size=(M, n))
    else:
        aw, ab = sqrt(3.0/d), sqrt(3.0)
        w1 = rng.uniform(-aw, aw, size=(M, d))
        b  = rng.uniform(-ab, ab, size=(M, n))
    p2 = np.sum(w1*w1, axis=1) * np.prod(b*b, axis=1)
    return float(np.mean(p2**a))

rng = np.random.default_rng(SEED)
ok = True
for kind in KINDS:
    for d in (2, 4, 6):
        for n in (1, 3, 5):
            F  = product_factor(kind, d, n, rng)
            Fm = mc_product(kind, d, n)
            tag = "PASS" if abs(F - Fm) < 5e-3 else "FAIL"
            if tag == "FAIL": ok = False
            print(f"{tag}  {kind:8s} d={d} n={n}: factor={F:.4f}  mc={Fm:.4f}")
print("\nALL CHECKS PASS" if ok else "\nCHECKS FAILED")

PASS  gaussian d=2 n=1: factor=0.7071  mc=0.7065
PASS  gaussian d=2 n=3: factor=0.5847  mc=0.5835
PASS  gaussian d=2 n=5: factor=0.5483  mc=0.5453
PASS  gaussian d=4 n=1: factor=0.7500  mc=0.7494
PASS  gaussian d=4 n=3: factor=0.6084  mc=0.6072
PASS  gaussian d=4 n=5: factor=0.5642  mc=0.5652
PASS  gaussian d=6 n=1: factor=0.7655  mc=0.7647
PASS  gaussian d=6 n=3: factor=0.6172  mc=0.6166
PASS  gaussian d=6 n=5: factor=0.5701  mc=0.5705
PASS  uniform  d=2 n=1: factor=0.8117  mc=0.8111
PASS  uniform  d=2 n=3: factor=0.7254  mc=0.7247
PASS  uniform  d=2 n=5: factor=0.6989  mc=0.6999
PASS  uniform  d=4 n=1: factor=0.8414  mc=0.8415
PASS  uniform  d=4 n=3: factor=0.7427  mc=0.7434
PASS  uniform  d=4 n=5: factor=0.7106  mc=0.7125
PASS  uniform  d=6 n=1: factor=0.8503  mc=0.8503
PASS  uniform  d=6 n=3: factor=0.7478  mc=0.7479
PASS  uniform  d=6 n=5: factor=0.7142  mc=0.7147

ALL CHECKS PASS


## Build the tables

For each $(kind, d, n)$ we record the scalar factor, the first-layer factor, the product $\mathbb{E}[(\|p\|^2)^\alpha]$, and the normalized gap $1-\mathbb{E}[(\|p\|^2)^\alpha]$.

In [10]:
results = {}
rng = np.random.default_rng(SEED)
for kind in KINDS:
    results[kind] = {}
    for d in WIDTHS:
        results[kind][str(d)] = {}
        for n in DEPTHS:
            a = n/(n+1)
            scalar = Eb2a(kind, a)
            first  = EWa(kind, d, a, rng)
            prod   = first * scalar**n
            results[kind][str(d)][str(n)] = {
                "alpha":        a,
                "scalar_factor": scalar,        # E[b^{2a}]
                "scalar_pow_n":  scalar**n,     # (E[b^{2a}])^n
                "first_factor":  first,         # E[(||w1||^2)^a]
                "product":       prod,          # E[(||p||^2)^a]
                "gap_norm":      1.0 - prod,    # normalized gap
                "gap":           (1+n)*(1.0 - prod),  # absolute gap (final - init)
                "init_sharp":    (1+n)*prod,    # predicted mean initial sharpness
                "final_sharp":   float(1+n),    # predicted final sharpness
            }
print("done")

done


In [11]:
def show(field, kind, fmt="{:.3f}", label=None):
    print(f"{label or field}  ({kind})")
    print("        " + "".join(f"   n={n}" for n in DEPTHS))
    for d in WIDTHS:
        row = "  ".join(fmt.format(results[kind][str(d)][str(n)][field]) for n in DEPTHS)
        print(f"  d={d}   {row}")
    print()

for kind in KINDS:
    show("scalar_factor", kind, label="scalar factor  E[b^2a]")
    show("product",       kind, label="product  E[(||p||^2)^a]")
    show("gap",           kind, label="gap = (1+n)(1 - product)")

scalar factor  E[b^2a]  (gaussian)
           n=1   n=2   n=3   n=4   n=5
  d=2   0.798  0.831  0.860  0.882  0.898
  d=3   0.798  0.831  0.860  0.882  0.898
  d=4   0.798  0.831  0.860  0.882  0.898
  d=5   0.798  0.831  0.860  0.882  0.898
  d=6   0.798  0.831  0.860  0.882  0.898

product  E[(||p||^2)^a]  (gaussian)
           n=1   n=2   n=3   n=4   n=5
  d=2   0.707  0.623  0.585  0.563  0.548
  d=3   0.735  0.643  0.600  0.575  0.559
  d=4   0.750  0.654  0.608  0.582  0.564
  d=5   0.759  0.661  0.614  0.586  0.568
  d=6   0.765  0.666  0.617  0.589  0.570

gap = (1+n)(1 - product)  (gaussian)
           n=1   n=2   n=3   n=4   n=5
  d=2   0.586  1.130  1.661  2.187  2.710
  d=3   0.530  1.070  1.600  2.125  2.648
  d=4   0.500  1.037  1.567  2.092  2.615
  d=5   0.482  1.017  1.546  2.071  2.594
  d=6   0.469  1.003  1.531  2.056  2.579

scalar factor  E[b^2a]  (uniform)
           n=1   n=2   n=3   n=4   n=5
  d=2   0.866  0.891  0.912  0.926  0.937
  d=3   0.866  0.891  0.912

## Write `sharpness_gap.json`

Same shape as `ps_stats.json` (`results[kind][d][n]`) so the site's table renderer can consume it the same way.

In [12]:
out = {
    "meta": {
        "widths": WIDTHS, "depths": DEPTHS, "kinds": KINDS,
        "mc_samples": MC_SAMPLES,
        "note": "C_i=0 approximation: init sharpness factor and depth gap. "
                "alpha=n/(n+1); product=E[(||p||^2)^alpha]; gap=(1+n)(1-product).",
        "fields": ["alpha", "scalar_factor", "scalar_pow_n", "first_factor",
                    "product", "gap_norm", "gap", "init_sharp", "final_sharp"],
    },
    "results": results,
}
with open("sharpness_gap.json", "w") as f:
    json.dump(out, f, indent=2)
print("wrote sharpness_gap.json")

wrote sharpness_gap.json
